In [35]:
import sys
import logging
import inspect

import numpy as np
np.random.seed(0)
import scipy.optimize
import gymnasium as gym

logging.basicConfig(level=logging.INFO,
        format='%(asctime)s [%(levelname)s] %(message)s',
        stream=sys.stdout, datefmt='%H:%M:%S')

In [36]:
env = gym.make('CliffWalking-v1')
env = env.unwrapped
for key in vars(env):
    logging.info('%s: %s', key, vars(env)[key])
logging.info('type = %s', inspect.getmro(type(env)))

11:54:45 [INFO] shape: (4, 12)
11:54:45 [INFO] start_state_index: 36
11:54:45 [INFO] nS: 48
11:54:45 [INFO] nA: 4
11:54:45 [INFO] is_slippery: False
11:54:45 [INFO] _cliff: [[False False False False False False False False False False False False]
 [False False False False False False False False False False False False]
 [False False False False False False False False False False False False]
 [False  True  True  True  True  True  True  True  True  True  True False]]
11:54:45 [INFO] P: {0: {0: [(1.0, np.int64(0), -1, False)], 1: [(1.0, np.int64(1), -1, False)], 2: [(1.0, np.int64(12), -1, False)], 3: [(1.0, np.int64(0), -1, False)]}, 1: {0: [(1.0, np.int64(1), -1, False)], 1: [(1.0, np.int64(2), -1, False)], 2: [(1.0, np.int64(13), -1, False)], 3: [(1.0, np.int64(0), -1, False)]}, 2: {0: [(1.0, np.int64(2), -1, False)], 1: [(1.0, np.int64(3), -1, False)], 2: [(1.0, np.int64(14), -1, False)], 3: [(1.0, np.int64(1), -1, False)]}, 3: {0: [(1.0, np.int64(3), -1, False)], 1: [(1.0, np.int

In [37]:
def evaluate_bellman(env, policy, gamma=1., verbose=True):
    if verbose:
        logging.info('policy = %s', policy)
    a, b = np.eye(env.nS), np.zeros((env.nS))
    for state in range(env.nS - 1):
        for action in range(env.nA):
            pi = policy[state][action]
            for p, next_state, reward, terminated in env.P[state][action]:
                a[state, next_state] -= (pi * gamma * p)
                b[state] += (pi * reward * p)
    v = np.linalg.solve(a, b)
    q = np.zeros((env.nS, env.nA))
    for state in range(env.nS - 1):
        for action in range(env.nA):
            for p, next_state, reward, terminated in env.P[state][action]:
                q[state][action] += ((reward + gamma * v[next_state]) * p)
    if verbose:
        logging.info('state values = %s', v)
        logging.info('action values = %s', q)
    return v, q

In [38]:
policy = np.ones((env.nS, env.nA)) / env.nA
state_values, action_values = evaluate_bellman(env, policy)

11:54:45 [INFO] policy = [[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.

In [39]:
actions = np.ones(env.nS, dtype=int)
actions[36] = 0
actions[11::12] = 2
policy = np.eye(env.nA)[actions]
state_values, action_values = evaluate_bellman(env, policy)

11:55:18 [INFO] policy = [[0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]
11:55:18 [INFO] state values = [ -14.  -13.  -12.  -11.  -10.   -9.   -8.   -7.   -6.   -5.   -4.   -3.
  -13.  -12.  -11.  -10.   -9.   -8.   -7.   -6.   -5.   -4.   -3.   -2.
  -12.  -11.  -10.   -9.   -8.   -7.   -6.   -5.   -4.   -3.   -2.   -1.
  -1

In [40]:
policy = np.random.uniform(size=(env.nS, env.nA))
policy = policy / np.sum(policy, axis=1, keepdims=True)
state_values, action_values = evaluate_bellman(env, policy)

11:56:10 [INFO] policy = [[0.2275677  0.29655611 0.24993822 0.22593797]
 [0.1766031  0.26924493 0.18241092 0.37174105]
 [0.36123028 0.14373357 0.29677919 0.19825697]
 [0.34389291 0.56035414 0.04300507 0.05274788]
 [0.0080841  0.33291382 0.31113736 0.34786472]
 [0.32406883 0.26464084 0.15281859 0.25847173]
 [0.0640631  0.34661191 0.07764701 0.51167798]
 [0.26418693 0.20992357 0.13393189 0.39195761]
 [0.27462234 0.34222196 0.01131228 0.37184343]
 [0.21442448 0.21611939 0.33060629 0.23884984]
 [0.23128455 0.2811586  0.4488116  0.03874524]
 [0.39766289 0.39997167 0.12547318 0.07689227]
 [0.18687207 0.21547646 0.33780682 0.25984466]
 [0.67668801 0.06986476 0.14300702 0.11044021]
 [0.40386721 0.15662972 0.28835589 0.15114718]
 [0.14942755 0.10374995 0.61693388 0.12988862]
 [0.1325213  0.24856725 0.55345295 0.0654585 ]
 [0.35220289 0.04039184 0.41042298 0.19698229]
 [0.41387165 0.25628418 0.31323958 0.01660459]
 [0.34578413 0.14696266 0.36208649 0.14516673]
 [0.21357411 0.27824066 0.04308481 

In [43]:
def optimal_bellman(env, gamma=1., verbose=True):
    p = np.zeros((env.nS, env.nA, env.nS))
    r = np.zeros((env.nS, env.nA))
    for state in range(env.nS - 1):
        for action in range(env.nA):
            for prob, next_state, reward, terminated in env.P[state][action]:
                p[state, action, next_state] += prob
                r[state, action] += (reward * prob)
    c = np.ones(env.nS)
    a_ub = gamma * p.reshape(-1, env.nS) - \
            np.repeat(np.eye(env.nS), env.nA, axis=0)
    b_ub = -r.reshape(-1)
    a_eq = np.zeros((0, env.nS))
    b_eq = np.zeros(0)
    bounds = [(None, None),] * env.nS
    res = scipy.optimize.linprog(c, a_ub, b_ub, bounds=bounds,
            method='highs-ipm')
    v = res.x
    q = r + gamma * np.dot(p, v)
    if verbose:
        logging.info('optimal state values = %s', v)
        logging.info('optimal action values = %s', q)
    return v, q

In [44]:
optimal_state_values, optimal_action_values = optimal_bellman(env)

11:56:51 [INFO] optimal state values = [-14. -13. -12. -11. -10.  -9.  -8.  -7.  -6.  -5.  -4.  -3. -13. -12.
 -11. -10.  -9.  -8.  -7.  -6.  -5.  -4.  -3.  -2. -12. -11. -10.  -9.
  -8.  -7.  -6.  -5.  -4.  -3.  -2.  -1. -13. -12. -11. -10.  -9.  -8.
  -7.  -6.  -5.  -4.  -1.  -0.]
11:56:51 [INFO] optimal action values = [[ -15.  -14.  -14.  -15.]
 [ -14.  -13.  -13.  -15.]
 [ -13.  -12.  -12.  -14.]
 [ -12.  -11.  -11.  -13.]
 [ -11.  -10.  -10.  -12.]
 [ -10.   -9.   -9.  -11.]
 [  -9.   -8.   -8.  -10.]
 [  -8.   -7.   -7.   -9.]
 [  -7.   -6.   -6.   -8.]
 [  -6.   -5.   -5.   -7.]
 [  -5.   -4.   -4.   -6.]
 [  -4.   -4.   -3.   -5.]
 [ -15.  -13.  -13.  -14.]
 [ -14.  -12.  -12.  -14.]
 [ -13.  -11.  -11.  -13.]
 [ -12.  -10.  -10.  -12.]
 [ -11.   -9.   -9.  -11.]
 [ -10.   -8.   -8.  -10.]
 [  -9.   -7.   -7.   -9.]
 [  -8.   -6.   -6.   -8.]
 [  -7.   -5.   -5.   -7.]
 [  -6.   -4.   -4.   -6.]
 [  -5.   -3.   -3.   -5.]
 [  -4.   -3.   -2.   -4.]
 [ -14.  -12.  -14.  -13.]
 

In [45]:
optimal_actions = optimal_action_values.argmax(axis=1)
logging.info('optimal policy = %s', optimal_actions)

11:57:02 [INFO] optimal policy = [1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 2 0
 0 0 0 0 0 0 0 0 0 1 0]
